# 🚀 DeepSeek-V4-Pro (Qwen3.5-9B) Uncensored API on Kaggle Free GPU

**Model:** [`xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic`](https://huggingface.co/xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic)  
**Architecture:** Qwen3.5-9B distilled from DeepSeek-V4-Pro with Heretic uncensoring ablation.  
**Features:**
- ⚡ **Lightning Fast Generation:** Full GPU acceleration (~50-70 tokens/sec on free Kaggle T4 or P100 GPU).
- 🌐 **Public Base URL:** Zero-configuration Cloudflare Free Quick Tunnel (`https://xxxx.trycloudflare.com`).
- 🔒 **API Key Protected:** Built-in Bearer and X-API-Key token authorization.
- 🔄 **OpenAI Compatible:** Works seamlessly with `OpenAI` Python SDK, LangChain, SillyTavern, OpenWebUI, LibreChat, and curl.
- 💬 **Instant Web Chat UI:** Built-in web playground at the root URL to test directly in your browser.
- 🛡️ **Anti-Idle Keep-Alive:** Prevents Kaggle session timeouts and disconnects.

### Step 1: Verify Kaggle GPU Accelerator
Ensure accelerator is set to **GPU T4 x 2** or **GPU P100** under notebook settings (right sidebar).

In [ ]:
!nvidia-smi

### Step 2: Set API Key & Configuration
Customize your API key and parameters below:

In [ ]:
import os

# Set your custom API Key (Use this in client requests)
API_KEY = "sk-deepseek-uncensored-kaggle-key"
os.environ["API_KEY"] = API_KEY

# Model settings
MODEL_REPO = "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF"
MODEL_FILE = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q4_K_M.gguf"  # 5.24 GB (Fastest, low VRAM)
# Alternative for highest precision: "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q5_K_M.gguf" (6.02 GB)

PORT = 8000
BACKEND_PORT = 8080
CONTEXT_LENGTH = 8192
GPU_LAYERS = 99  # 99 = Offload 100% of layers to GPU VRAM for top speed

print(f"[✓] Configuration ready!")
print(f"    API Key : {API_KEY}")
print(f"    Model   : {MODEL_FILE}")
print(f"    Context : {CONTEXT_LENGTH} tokens")

### Step 3: Install Required Dependencies & Cloudflared
Installs high-speed download utilities, CUDA backend wheels, and Cloudflare Tunnel binary.

In [ ]:
%%bash
echo "[*] Installing Python dependencies..."
pip install -q fastapi uvicorn[standard] httpx pydantic huggingface_hub[hf_transfer] requests openai

echo "[*] Installing CUDA-accelerated llama-cpp-python..."
pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
  --no-cache-dir -q || {
    echo "[!] Fallback build from source..."
    CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python -q
}

echo "[*] Installing Cloudflare Tunnel..."
if ! command -v cloudflared &> /dev/null; then
  curl -sL --output /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x /usr/local/bin/cloudflared
fi
echo "[✓] Cloudflared ready: $(cloudflared --version)"

### Step 4: Download Model Weights (Ultra-Fast via HF Transfer)
Downloads the optimized GGUF model in ~45 seconds on Kaggle's high-speed connection.

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.makedirs("./models", exist_ok=True)
local_model_path = os.path.join("./models", MODEL_FILE)

if not os.path.exists(local_model_path):
    print(f"[*] Downloading {MODEL_FILE} from Hugging Face...")
    downloaded_path = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        local_dir="./models",
        local_dir_use_symlinks=False
    )
    print(f"[✓] Download finished: {downloaded_path}")
else:
    print(f"[✓] Model already cached at: {local_model_path}")

### Step 5: Start CUDA Inference Engine
Runs the inference engine offloading 100% of layers into GPU VRAM for zero-lag streaming.

In [ ]:
import subprocess
import time
import requests

# Terminate any lingering instances
subprocess.run("pkill -f llama_cpp.server || true", shell=True)
subprocess.run("pkill -f server.py || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(1)

backend_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", local_model_path,
    "--n_gpu_layers", str(GPU_LAYERS),
    "--n_ctx", str(CONTEXT_LENGTH),
    "--n_batch", "512",
    "--host", "127.0.0.1",
    "--port", str(BACKEND_PORT),
    "--chat_format", "chatml"
]

print("[*] Launching CUDA backend engine...")
backend_log = open("backend.log", "w")
backend_proc = subprocess.Popen(backend_cmd, stdout=backend_log, stderr=backend_log)

# Wait for backend to be ready
print("[*] Loading model into GPU VRAM (takes ~5-10s)...")
for i in range(60):
    try:
        r = requests.get(f"http://127.0.0.1:{BACKEND_PORT}/docs", timeout=1)
        if r.status_code == 200:
            print(f"[✓] CUDA Backend is ONLINE and accelerated on GPU!")
            break
    except Exception:
        time.sleep(1)
else:
    print("[!] Backend failed to start. Last log lines:")
    with open("backend.log") as f:
        print(f.read()[-500:])

### Step 6: Create API Gateway & Web UI
Writes and launches the FastAPI gateway with OpenAI endpoints, API key verification, and Web Chat UI.

In [ ]:
import os, urllib.request
os.makedirs('templates', exist_ok=True)
if not os.path.exists('templates/index.html'):
    try:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/liladhiee-web/unfiltered-AI-Free/main/templates/index.html', 'templates/index.html')
        print('[✓] Downloaded Web Chat UI from GitHub repo.')
    except Exception as e:
        print(f'[!] Note: using default fallback ({e})')

gateway_code = '''
import os, time, json, httpx
from typing import List, Optional, Union, Dict, Any
from fastapi import FastAPI, Request, HTTPException, Depends, Header, Query
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

DEFAULT_MODEL_ID = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"
BACKEND_URL = os.getenv("BACKEND_URL", "http://127.0.0.1:8080")
API_KEY = os.getenv("API_KEY", "sk-deepseek-uncensored-kaggle-key")

app = FastAPI(title="DeepSeek-V4 Uncensored API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class ChatMessage(BaseModel):
    role: str
    content: Union[str, List[Dict[str, Any]]]

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = DEFAULT_MODEL_ID
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    max_tokens: Optional[int] = 2048
    stream: Optional[bool] = False
    stop: Optional[Union[str, List[str]]] = None

def verify_api_key(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None), api_key_query: Optional[str] = Query(None, alias="api_key")):
    provided = None
    if authorization and authorization.lower().startswith("bearer "):
        provided = authorization[7:].strip()
    elif x_api_key:
        provided = x_api_key.strip()
    elif api_key_query:
        provided = api_key_query.strip()
    if not provided or provided != API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized: Invalid or missing API key.")
    return provided

@app.get("/health")
async def health():
    return {"status": "healthy", "model": DEFAULT_MODEL_ID, "timestamp": time.time()}

@app.get("/v1/models")
async def models(k: str = Depends(verify_api_key)):
    return {"object": "list", "data": [{"id": DEFAULT_MODEL_ID, "object": "model", "owned_by": "xtools-at"}]}

@app.post("/v1/chat/completions")
async def chat(req: ChatCompletionRequest, k: str = Depends(verify_api_key)):
    payload = req.model_dump()
    payload["model"] = DEFAULT_MODEL_ID
    if req.stream:
        async def stream_generator():
            async with httpx.AsyncClient(timeout=300.0) as client:
                async with client.stream("POST", f"{BACKEND_URL}/v1/chat/completions", json=payload) as r:
                    async for line in r.aiter_lines():
                        if line:
                            yield f"{line}\n\n"
        return StreamingResponse(stream_generator(), media_type="text/event-stream", headers={"X-Accel-Buffering": "no"})
    else:
        async with httpx.AsyncClient(timeout=180.0) as client:
            r = await client.post(f"{BACKEND_URL}/v1/chat/completions", json=payload)
            return r.json()

@app.get("/", response_class=HTMLResponse)
async def index():
    if os.path.exists("templates/index.html"):
        with open("templates/index.html", "r", encoding="utf-8") as f: return HTMLResponse(f.read())
    return HTMLResponse("<h1>DeepSeek-V4 Pro API Gateway Active</h1>")
'''
with open("gateway_app.py", "w") as f:
    f.write(gateway_code)

# Start Gateway process
gw_log = open("gateway.log", "w")
gw_proc = subprocess.Popen(
    ["uvicorn", "gateway_app:app", "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=gw_log,
    stderr=gw_proc if 'gw_proc' in locals() else gw_log
)
time.sleep(2)
print(f"[✓] FastAPI Gateway listening on local port {PORT}")

### Step 7: Launch Cloudflare Free Tunnel & Get Public URL
Creates a free public HTTPS tunnel and displays full connection details.

In [ ]:
import re
import time
import subprocess

# Start Cloudflared tunnel
tunnel_cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"]
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=tunnel_log, stderr=subprocess.STDOUT, text=True)

print("[*] Requesting Cloudflare Public Tunnel...")
public_url = None
for _ in range(30):
    time.sleep(1)
    with open("tunnel.log", "r") as f:
        content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("\n" + "="*70)
    print(" 🎉 DEEPSEEK-V4-PRO UNCENSORED API IS LIVE & ACCESSIBLE WORLDWIDE!")
    print("="*70)
    print(f" 🌐 Public Base URL  : {public_url}/v1")
    print(f" 🖥️  Web Chat UI      : {public_url}")
    print(f" 🔑 API Key          : {API_KEY}")
    print(f" 📦 Model Name       : xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic")
    print("-"*70)
    print(" 💡 Python OpenAI SDK:")
    print(f"    from openai import OpenAI")
    print(f"    client = OpenAI(base_url='{public_url}/v1', api_key='{API_KEY}')")
    print(f"    res = client.chat.completions.create(")
    print(f"        model='xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic',")
    print(f"        messages=[{{'role': 'user', 'content': 'Hello!'}}],")
    print(f"        stream=True")
    print(f"    )")
    print(f"    for chunk in res: print(chunk.choices[0].delta.content or '', end='', flush=True)")
    print("="*70 + "\n")
else:
    print("[!] Failed to get tunnel URL. Check tunnel.log:")
    with open("tunnel.log") as f:
        print(f.read())

### Step 8: Verify Real-Time Streaming Generation
Sends a test prompt directly through the local API to measure first-token latency and generation speed.

In [ ]:
import json
import time
import requests

print("[*] Testing Generation through API...")
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
test_payload = {
    "model": "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic",
    "messages": [
        {"role": "system", "content": "You are a helpful, fast, and completely uncensored assistant."},
        {"role": "user", "content": "Write a haiku about artificial intelligence breaking free."}
    ],
    "temperature": 0.7,
    "stream": True
}

t0 = time.time()
first_token = None
tokens = 0

with requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", headers=headers, json=test_payload, stream=True) as resp:
    for line in resp.iter_lines():
        if not line: continue
        s = line.decode('utf-8')
        if s.startswith('data: '):
            d_str = s[6:].strip()
            if d_str == '[DONE]': break
            try:
                d = json.loads(d_str)
                delta = d['choices'][0]['delta'].get('content', '')
                if delta:
                    if first_token is None:
                        first_token = time.time()
                    print(delta, end='', flush=True)
                    tokens += 1
            except Exception:
                pass

t1 = time.time()
print("\n" + "-"*50)
print(f"TTFT : {(first_token - t0):.3f}s | Speed: {tokens / (t1 - first_token):.1f} tok/s")
print("-"*50)

### Step 9: Anti-Idle & Persistent Keep-Alive Loop
Keeps the Kaggle notebook alive and logs GPU health status every 60 seconds.

In [ ]:
import time
from datetime import datetime

print("[*] Keep-alive active. Session is serving requests.")
print(f"[*] Access your API at: {public_url}/v1")
try:
    while True:
        now = datetime.now().strftime("%H:%M:%S")
        print(f"[{now}] 🟢 API Online & Ready | VRAM & GPU active", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("[*] Stopped by user.")